# CS604 Colab launcher — Ana: merged-LoRA latency (no training)

Loads a finished LoRA run from `CS604_RUNS` (`RUN_NAME`), checks that merging the LoRA weights into the base weights leaves the predictions unchanged, and measures inference latency in one session for: the model as trained (unmerged), the merged model, and a plain ViT-B with the same head (reference). Same protocol as the other launchers: A100, bf16, synthetic 336 px inputs, 30 warm-ups + 200 timed iterations, batch 1 and 64.

No data download and no training: about 10 minutes. Writes `inference_latency_merged.json` into the run folder.

Runtime: **A100 GPU**. Use **Runtime → Run all**; the runtime ends itself at the end when `AUTO_DISCONNECT = True`.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
RUN_NAME = "lora_20261007_051926"  # the LoRA r24 run folder in CS604_RUNS
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Merge LoRA and measure latency

In [ ]:
import subprocess
import sys

RUN_DIR = f"{DRIVE_OUT}/{RUN_NAME}"
proc = subprocess.run([sys.executable, "-u", "scripts/latency_merged.py", "--run_dir", RUN_DIR])
if proc.returncode != 0:
    end_runtime(f"latency script failed (exit {proc.returncode}); see the output above")
    raise SystemExit("latency script failed")

## 5. Results

`equivalence_check` should show 0 LoRA layers left, a tiny max logit difference (rounding only) and top-1 agreement of 1.0. Merged latency should be close to the plain ViT.

In [ ]:
import json
from pathlib import Path

import pandas as pd

res = json.loads((Path(RUN_DIR) / "inference_latency_merged.json").read_text())
print(res["gpu"], "|", res["precision"], "|", res["img_size"], "px")
print("Equivalence check:", res["equivalence_check"])
rows = []
for name in ("unmerged", "merged", "plain_vit_same_head"):
    bs1, bs64 = res[name]
    rows.append({"model": name, "bs1 ms/image": round(bs1["ms_per_image"], 2), "bs64 ms/image": round(bs64["ms_per_image"], 3)})
display(pd.DataFrame(rows))

## 6. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")